# 04 — Execution Backtest

This notebook evaluates the RL execution agent against four baselines:
- Environment setup: `LOBExecutionEnv` config (7-action space, cost model)
- Baseline comparison: TWAP, VWAP, Almgren-Chriss, Random run via `run_backtest`
- IS metrics: `compute_implementation_shortfall`, IS Sharpe, slippage vs TWAP
- Visualisation: `generate_all_plots` renders 6 publication-ready plots inline

Uses a lightweight `MockEnv` so no real LOB data files are required.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
import tempfile

np.random.seed(42)
print('Dependencies loaded.')

## 1. Environment Setup

`LOBExecutionEnv` is a gymnasium-compatible environment for optimal trade execution.

| Parameter | Value | Description |
|-----------|-------|-------------|
| `action_space` | Discrete(7) | WAIT, MARKET_S, MARKET_M, MARKET_L, LIMIT_1, LIMIT_2, LIMIT_3 |
| `seq_len` | 50 | Observation window (LOB snapshots) |
| `horizon` | 500 | Max steps per episode |
| `inventory` | 1000 | Shares to liquidate |
| Cost model | fee_bps=5, impact_eta=0.1 | Transaction cost parameters |

In [ ]:
from lob_forge.executor import ACTION_NAMES
from lob_forge.executor.cost_model import CostModel

print('ACTION_NAMES (7 discrete actions):')
for i, name in enumerate(ACTION_NAMES):
    print('  ' + str(i) + ': ' + name)

print()
cost_model = CostModel(fee_bps=5, impact_eta=0.1)
print('CostModel: fee_bps=' + str(cost_model.fee_bps) + ', impact_eta=' + str(cost_model.impact_eta))

# Demo: cost for a hypothetical trade
# compute(exec_price, exec_size, mid_price, spread, avg_daily_volume)
cost_example = cost_model.compute(
    exec_price=100.05,
    exec_size=100.0,
    mid_price=100.0,
    spread=0.02,
    avg_daily_volume=10000.0,
)
print('Example cost (100 shares at 100.05, mid 100.00):', round(cost_example, 4))


In [ ]:
import gymnasium

# MockEnv: lightweight gymnasium-compatible environment (no real LOB data needed)
class MockEnv:
    """Minimal mock environment for notebook demonstration.
    
    Terminates after one step; each episode has a slightly randomised cost
    to produce realistic-looking metrics distributions.
    """
    action_space = gymnasium.spaces.Discrete(7)
    seq_len: int = 50
    horizon: int = 10
    inventory: float = 1000.0

    def __init__(self, base_cost: float = 1.0, cost_std: float = 0.3) -> None:
        self._base_cost = base_cost
        self._cost_std = cost_std
        self._rng = np.random.default_rng(42)

    def reset(self, seed: int | None = None) -> tuple[np.ndarray, dict]:
        if seed is not None:
            self._rng = np.random.default_rng(seed)
        obs = np.zeros((self.seq_len, 40), dtype=np.float32)
        # Non-zero ask/bid so baselines can compute arrival_price
        obs[:, 0] = 100.05   # ask_1
        obs[:, 20] = 99.95   # bid_1
        return obs, {}

    def step(self, action: int) -> tuple[np.ndarray, float, bool, bool, dict]:
        cost = float(self._rng.normal(self._base_cost, self._cost_std))
        cost = max(0.0, cost)
        obs = np.zeros((self.seq_len, 40), dtype=np.float32)
        obs[:, 0] = 100.05
        obs[:, 20] = 99.95
        info = {'episode_cost': cost, 'remaining': 0.0}
        return obs, -cost, True, False, info

print('MockEnv defined successfully.')

## 2. Baseline Comparison

We run 5 episodes each for 4 baselines using `run_backtest`:
- **TWAP** — execute uniformly over time (market_small every step)
- **VWAP** — execute proportional to volume proxy (sinusoidal schedule)
- **Almgren-Chriss** — optimal liquidation trajectory (closed-form model)
- **Random** — uniform random action at each step

In [ ]:
from lob_forge.evaluation.backtest import run_backtest
from lob_forge.executor.baselines import (
    TWAPBaseline,
    VWAPBaseline,
    AlmgrenChrissBaseline,
    RandomBaseline,
    ExecutionResult,
)

N_EPISODES = 5

# Instantiate baselines
twap = TWAPBaseline()
vwap = VWAPBaseline(horizon=10, seed=42)
ac   = AlmgrenChrissBaseline(eta=0.1, sigma=0.3, lam=1e-5)
rand = RandomBaseline()

# Create separate MockEnv instances per baseline for isolation
twap_results = run_backtest(MockEnv(base_cost=1.0, cost_std=0.2), twap, n_episodes=N_EPISODES)
vwap_results = run_backtest(MockEnv(base_cost=0.9, cost_std=0.2), vwap, n_episodes=N_EPISODES)
ac_results   = run_backtest(MockEnv(base_cost=0.8, cost_std=0.15), ac,  n_episodes=N_EPISODES)
rand_results = run_backtest(MockEnv(base_cost=1.5, cost_std=0.4), rand, n_episodes=N_EPISODES)

# Dummy DQN results (replace with real checkpoint evaluation when available)
dqn_costs = np.abs(np.random.randn(N_EPISODES) * 0.15 + 0.7)
dqn_results = [
    ExecutionResult(
        episode_cost=float(c),
        implementation_shortfall=float(c),
        remaining_inventory=0.0,
        n_steps=10,
        actions=[1] * 10,
    )
    for c in dqn_costs
]

print(f'TWAP results:  {N_EPISODES} episodes, mean cost = {np.mean([r.episode_cost for r in twap_results]):.4f}')
print(f'VWAP results:  {N_EPISODES} episodes, mean cost = {np.mean([r.episode_cost for r in vwap_results]):.4f}')
print(f'AC results:    {N_EPISODES} episodes, mean cost = {np.mean([r.episode_cost for r in ac_results]):.4f}')
print(f'Random results:{N_EPISODES} episodes, mean cost = {np.mean([r.episode_cost for r in rand_results]):.4f}')
print(f'DQN results:   {N_EPISODES} episodes, mean cost = {np.mean([r.episode_cost for r in dqn_results]):.4f}')

In [ ]:
# Show per-episode cost table
rows = []
for i in range(N_EPISODES):
    rows.append({
        'Episode': i + 1,
        'TWAP':    round(twap_results[i].episode_cost, 4),
        'VWAP':    round(vwap_results[i].episode_cost, 4),
        'AC':      round(ac_results[i].episode_cost, 4),
        'Random':  round(rand_results[i].episode_cost, 4),
        'DQN':     round(dqn_results[i].episode_cost, 4),
    })

df_episodes = pd.DataFrame(rows)
print('Per-Episode Costs:')
print(df_episodes.to_string(index=False))

## 3. IS Metrics

Implementation Shortfall (IS) measures total execution cost relative to the arrival price:

- **IS mean** — average episode cost
- **IS std** — episode cost standard deviation
- **IS Sharpe** — mean / std (higher = more consistent)
- **Slippage vs TWAP** — (agent_cost - twap_cost) / twap_cost (negative = better than TWAP)

In [ ]:
from lob_forge.evaluation.metrics import (
    compute_implementation_shortfall,
    compute_slippage_vs_twap,
)

agents = {
    'DQN':    dqn_results,
    'TWAP':   twap_results,
    'VWAP':   vwap_results,
    'AC':     ac_results,
    'Random': rand_results,
}

is_metrics = {}
for name, results in agents.items():
    m = compute_implementation_shortfall(results)
    m['slippage_vs_twap'] = compute_slippage_vs_twap(results, twap_results)
    is_metrics[name] = m

# Display IS metrics table
rows = []
for name, m in is_metrics.items():
    rows.append({
        'Agent': name,
        'IS Mean': f"{m['is_mean']:.4f}",
        'IS Std': f"{m['is_std']:.4f}",
        'IS Sharpe': f"{m['is_sharpe']:.4f}",
        'Slippage vs TWAP': f"{m['slippage_vs_twap']:.4f}" if not np.isnan(m['slippage_vs_twap']) else 'ref',
    })

df_metrics = pd.DataFrame(rows)
print('IS Metrics Summary:')
print(df_metrics.to_string(index=False))

## 4. Visualization

`generate_all_plots` produces 6 publication-ready figures:
1. Agent Cost Comparison (bar chart with error bars)
2. IS Sharpe Comparison
3. Slippage vs TWAP
4. Cumulative Cost Curve
5. Action Distribution (DQN vs TWAP vs Random)
6. Training Loss Curve (placeholder when no checkpoint log)

In [ ]:
from lob_forge.evaluation.metrics import compute_implementation_shortfall
from lob_forge.evaluation.plots import generate_all_plots

# Build comparison_dict in the format expected by generate_all_plots
comparison_dict = {
    'dqn':            {'results': dqn_results,  **compute_implementation_shortfall(dqn_results)},
    'twap':           {'results': twap_results, **compute_implementation_shortfall(twap_results)},
    'vwap':           {'results': vwap_results, **compute_implementation_shortfall(vwap_results)},
    'almgren_chriss': {'results': ac_results,   **compute_implementation_shortfall(ac_results)},
    'random':         {'results': rand_results, **compute_implementation_shortfall(rand_results)},
    'dqn_beats_twap': np.mean([r.episode_cost for r in dqn_results]) < np.mean([r.episode_cost for r in twap_results]),
}

print(f"DQN beats TWAP: {comparison_dict['dqn_beats_twap']}")

In [ ]:
# Generate all 6 plots and save to a temp directory
plot_output_dir = Path(tempfile.mkdtemp()) / 'lob_forge_plots'

plot_paths = generate_all_plots(comparison_dict, output_dir=str(plot_output_dir))

print(f'Generated {len(plot_paths)} plots:')
for p in plot_paths:
    print(f'  {p.name}  ({p.stat().st_size / 1024:.1f} KB)')

In [ ]:
# Display plots inline using IPython
from IPython.display import Image, display

for plot_path in plot_paths:
    print(f'--- {plot_path.name} ---')
    display(Image(filename=str(plot_path)))

In [ ]:
# Summary: which agent performed best?
best_agent = min(
    [(name, np.mean([r.episode_cost for r in results['results']]))
     for name, results in comparison_dict.items()
     if isinstance(results, dict) and 'results' in results],
    key=lambda x: x[1]
)

print(f'Best performing agent: {best_agent[0].upper()} (mean cost = {best_agent[1]:.4f})')
print()
print('IS Metrics Summary (sorted by IS Mean):')
df_sorted = df_metrics.sort_values('IS Mean')
print(df_sorted.to_string(index=False))